# Week 3 – Exploratory Data Analysis & Feature Engineering  
## Predictive Maintenance for IT Infrastructure  

In this notebook, we analyze system behavior and prepare advanced features for failure prediction.

Objectives:
- Perform exploratory data analysis (EDA)  
- Identify patterns leading to failures  
- Create engineered features  
- Prepare time-series data for deep learning models (LSTM)  

Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

Load Clean Dataset

In [ ]:
df = pd.read_csv("clean_dataset.csv")

df.head()

##PART 1 – EXPLORATORY DATA ANALYSIS
###CPU Usage vs Time

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df.index, df["CPU_Usage(%)"])
plt.title("CPU Usage Over Time")
plt.xlabel("Time")
plt.ylabel("CPU Usage")
plt.show()

### Memory Behavior Before Failures

In [ ]:
failure_data = df[df["failure"] == 1]

plt.figure(figsize=(10,5))
plt.hist(failure_data["Memory_Usage(%)"], bins=30)
plt.title("Memory Usage During Failures")
plt.xlabel("Memory Usage")
plt.ylabel("Frequency")
plt.show()

### Correlation Between Features

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

##PART 2 – PATTERN IDENTIFICATION
### Observing Pre-Failure Patterns

In [ ]:
# Compare normal vs failure CPU
plt.figure(figsize=(10,5))
sns.boxplot(x="failure", y="CPU_Usage(%)", data=df)
plt.title("CPU Usage vs Failure")
plt.show()

### Threshold Detection

In [ ]:
high_cpu_failures = df[(df["CPU_Usage(%)"] > 1) & (df["failure"] == 1)]

print("High CPU failures count:", len(high_cpu_failures))

## PART 3 – FEATURE ENGINEERING
### Rolling Features

In [ ]:
df["cpu_rolling_mean"] = df["CPU_Usage(%)"].rolling(window=5).mean()
df["memory_rolling_mean"] = df["Memory_Usage(%)"].rolling(window=5).mean()

### Lag Features (Previous Values)

In [ ]:
df["cpu_lag1"] = df["CPU_Usage(%)"].shift(1)
df["memory_lag1"] = df["Memory_Usage(%)"].shift(1)

### Failure Indicator

In [ ]:
df["high_cpu_flag"] = (df["CPU_Usage(%)"] > 1).astype(int)

### Handling Missing Values

In [ ]:
df.dropna(inplace=True)

### Save Feature Engineered Dataset

In [ ]:
df.to_csv("feature_engineered_dataset.csv", index=False)

## PART 4 – DEEP LEARNING PREPARATION (LSTM)
### Creating Time-Series Sequences

In [ ]:
sequence_length = 10

X_seq = []
y_seq = []

data = df.drop(columns=["failure"]).values
target = df["failure"].values

for i in range(len(data) - sequence_length):
    X_seq.append(data[i:i+sequence_length])
    y_seq.append(target[i+sequence_length])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print("X shape:", X_seq.shape)
print("y shape:", y_seq.shape)

### Save Sequence Data

In [ ]:
np.save("X_seq.npy", X_seq)
np.save("y_seq.npy", y_seq)